In [ ]:
from pathlib import Path
import os, json, random, hashlib, platform, importlib.metadata, time, math
import torch_directml
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# Change these before starting an experiment. Use a new RUN_NAME for each run.
DATA_ROOT = None  # auto-find dataset in current folder or one folder below
RUN_NAME = 'directml_finetune_v3'
SOURCE_RUN = Path('outputs/directml_sgd_v2')
MAX_GRAD_NORM = 1.0
OUTPUT_BASE = Path('outputs')
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 4    # reduce to 2 if GPU/shared memory is exhausted
NUM_WORKERS = 0   # Windows notebook: load images in the main process
EPOCHS = 6
WARMUP_EPOCHS = 0
PATIENCE = 4
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
WEIGHTS_PATH = None  # optional local torchvision state_dict (.pth)
SMOKE_TEST = False    # two epochs / two batches each: tests head-only AND fine-tuning

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch_directml.device()
AMP = False  # DirectML: use full float32; no autocast or GradScaler
print('Device:', DEVICE, '| torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('DirectML adapter:', torch_directml.device_name(0))
OUT = OUTPUT_BASE / RUN_NAME
assert (SOURCE_RUN/'best_model.pt').exists(), 'Missing baseline best_model.pt'
assert (SOURCE_RUN/'split_manifest.csv').exists(), 'Missing baseline split_manifest.csv'
OUT.mkdir(parents=True, exist_ok=False)  # protects earlier experimental evidence
config = {k: globals()[k] for k in ['RUN_NAME','SEED','IMAGE_SIZE','BATCH_SIZE','NUM_WORKERS','EPOCHS','WARMUP_EPOCHS','PATIENCE','BACKBONE_LR','HEAD_LR','WEIGHT_DECAY','SMOKE_TEST']}
config.update(optimizer='SGD', momentum=0.9, freeze_bn_statistics=True, device=str(DEVICE))
config.update(source_run=str(SOURCE_RUN), max_grad_norm=MAX_GRAD_NORM)
config['weights'] = str(SOURCE_RUN/'best_model.pt')
(OUT/'config.json').write_text(json.dumps(config, indent=2))
versions = {p: importlib.metadata.version(p) for p in ['torch','torchvision','torch-directml','numpy','pandas','Pillow','scikit-learn','matplotlib','tqdm']}
(OUT/'environment.json').write_text(json.dumps({'python':platform.python_version(), **versions}, indent=2))

In [ ]:
if DATA_ROOT is None:
    search_folders = [Path.cwd()] + [p for p in Path.cwd().iterdir() if p.is_dir()]
    candidates = sorted({p for p in search_folders if (p/'sample_submission.csv').exists()
                         and ((p/'train_labels.csv').exists() or (p/'train.csv').exists())})
    if len(candidates) != 1:
        raise ValueError(f'Set DATA_ROOT explicitly. Found candidate folders: {candidates}')
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)
label_file = DATA_ROOT/'train_labels.csv'
if not label_file.exists(): label_file = DATA_ROOT/'train.csv'
labels = pd.read_csv(label_file, dtype=str, keep_default_na=False)
sample = pd.read_csv(DATA_ROOT/'sample_submission.csv', dtype=str, keep_default_na=False)
for name, frame in [('training', labels), ('submission template', sample)]:
    if not {'image_id','class'}.issubset(frame.columns):
        raise ValueError(f'{name} columns are {list(frame.columns)}; expected image_id,class.')
    if frame.image_id.eq('').any() or frame.image_id.duplicated().any():
        raise ValueError(f'Empty or duplicated IDs in {name}.')
if labels['class'].eq('').any(): raise ValueError('Empty training labels.')
classes = sorted(labels['class'].unique().tolist())
assert len(classes) == 8, classes
assert len(sample) == 29443, f'Expected 29,443 test IDs; found {len(sample)}'
class_to_idx = {label:i for i,label in enumerate(classes)}
(OUT/'class_mapping.json').write_text(json.dumps(class_to_idx, indent=2))

def numeric_key(value):
    stem = Path(str(value)).stem
    return str(int(stem)) if stem.isdigit() else None

def resolve_images(folder, ids):
    files = sorted(p for p in folder.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
    if not files: raise FileNotFoundError(f'No images in {folder}')
    lookups = [defaultdict(list) for _ in range(4)]
    for p in files:
        keys = [p.relative_to(folder).as_posix(), p.name, p.stem, numeric_key(p.name)]
        for table, key in zip(lookups, keys):
            if key is not None: table[key].append(p)
    resolved=[]
    for value in ids:
        keys=[str(value), Path(value).name, Path(value).stem, numeric_key(value)]
        matches=[]
        for table,key in zip(lookups,keys):
            matches=table.get(key, [])
            if matches: break
        if len(matches)!=1: raise ValueError(f'{value}: {len(matches)} matching files in {folder}')
        resolved.append(str(matches[0]))
    if len(set(resolved)) != len(resolved): raise ValueError('Multiple IDs resolve to the same file.')
    return resolved

labels['path'] = resolve_images(DATA_ROOT/'train', labels.image_id)
labels['target'] = labels['class'].map(class_to_idx)
print('Root:', DATA_ROOT, '| training rows:',len(labels),'| test IDs:',len(sample))
print('Exact classes:', classes)
labels['class'].value_counts().reindex(classes).rename('count').to_csv(OUT/'class_counts.csv')
labels['class'].value_counts().reindex(classes).plot.bar(title='Training class counts', ylabel='Images')
plt.tight_layout(); plt.savefig(OUT/'class_counts.png', dpi=150); plt.show()

In [ ]:
# Reuse the original split; never put previously trained images in validation.
previous = pd.read_csv(SOURCE_RUN/'split_manifest.csv', dtype=str, keep_default_na=False)
assert previous.image_id.is_unique
assert set(previous.image_id).issubset(set(labels.image_id))
matched = labels.merge(previous[['image_id','class','split','image_hash']],
                       on='image_id', suffixes=('', '_previous'), validate='one_to_one')
assert matched['class'].equals(matched['class_previous']), 'Labels changed.'
assert set(matched.split) == {'train','validation'}
# Verify decoded training pixels still match the baseline audit.
for row in tqdm(matched.itertuples(), total=len(matched), desc='Verify original dataset'):
    with Image.open(row.path) as raw:
        im = ImageOps.exif_transpose(raw).convert('RGB')
        digest = hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
    assert digest == row.image_hash, f'Image changed: {row.image_id}'
labels = matched.drop(columns='class_previous')
train_df = labels[labels.split.eq('train')].reset_index(drop=True)
val_df = labels[labels.split.eq('validation')].reset_index(drop=True)
assert not set(train_df.image_hash) & set(val_df.image_hash)
assert set(train_df['class']) == set(val_df['class']) == set(classes)
labels.to_csv(OUT/'split_manifest.csv', index=False)
print(pd.crosstab(labels['class'], labels.split))

In [ ]:
mean=(0.485,0.456,0.406); std=(0.229,0.224,0.225)
eval_tf=T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),T.ToTensor(),T.Normalize(mean,std)])
train_tf=T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),T.RandomHorizontalFlip(),
                    T.RandomRotation(10),T.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.1),
                    T.ToTensor(),T.Normalize(mean,std)])
class FeatureDataset(Dataset):
    def __init__(self, frame, transform): self.frame=frame.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with Image.open(row.path) as raw: im=ImageOps.exif_transpose(raw).convert('RGB')
        return self.transform(im), int(row.target) if 'target' in self.frame.columns else -1

def seed_worker(worker_id):
    seed=torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)

def loader(frame, transform, shuffle=False):
    return DataLoader(FeatureDataset(frame,transform),batch_size=BATCH_SIZE,shuffle=shuffle,
                      num_workers=NUM_WORKERS,pin_memory=False,worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(SEED))
train_loader=loader(train_df,train_tf,True)
val_loader=loader(val_df,eval_tf)
source_checkpoint = torch.load(SOURCE_RUN/'best_model.pt', map_location='cpu', weights_only=True)
assert source_checkpoint['classes'] == classes
assert source_checkpoint['config']['IMAGE_SIZE'] == IMAGE_SIZE
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model.load_state_dict(source_checkpoint['state_dict'])
assert all(torch.isfinite(t).all().item() for t in model.state_dict().values())
model = model.to(DEVICE)
print('Starting validation accuracy:', source_checkpoint['val_accuracy'])
(OUT/'architecture.txt').write_text(str(model))
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD([{'params':model.features.parameters(),'lr':BACKBONE_LR},
                           {'params':model.classifier.parameters(),'lr':HEAD_LR}],
                          momentum=0.9, weight_decay=WEIGHT_DECAY, foreach=False)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=1)

In [ ]:
def run_epoch(data_loader, training=False, freeze_backbone=False):
    model.train(training)
    if training:
        if freeze_backbone:
            model.features.eval()
        else:
            # Keep pretrained running statistics with a small batch size.
            for module in model.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    module.eval()
    total_loss = correct = count = 0
    all_probs = []; all_targets = []
    start = time.perf_counter()
    with torch.set_grad_enabled(training):
        for batch_i, (images, targets_cpu) in enumerate(tqdm(data_loader, leave=False)):
            images = images.to(DEVICE)
            targets = targets_cpu.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            loss_value = loss.detach().cpu().item()
            if not math.isfinite(loss_value):
                raise RuntimeError(f'Non-finite loss at batch {batch_i}; best checkpoint is preserved.')
            if training:
                loss.backward()
                parameters = [p for p in model.parameters() if p.grad is not None]
                # CPU reductions avoid unsupported DirectML norm operations.
                norms = torch.stack([p.grad.detach().cpu().double().norm() for p in parameters])
                grad_norm = norms.norm().item()
                if not math.isfinite(grad_norm):
                    optimizer.zero_grad(set_to_none=True)
                    raise RuntimeError(f'Non-finite gradients at batch {batch_i}; stopped before update.')
                scale = min(1.0, MAX_GRAD_NORM / (grad_norm + 1e-12))
                if scale < 1.0:
                    for p in parameters:
                        p.grad.mul_(scale)
                optimizer.step()
            # Small metric tensors go to CPU; no unsupported GPU metric operations.
            logits_cpu = logits.detach().cpu()
            total_loss += loss_value * len(targets_cpu)
            correct += (logits_cpu.argmax(1) == targets_cpu).sum().item()
            count += len(targets_cpu)
            if not training:
                all_probs.append(logits_cpu.softmax(1).numpy())
                all_targets.append(targets_cpu.numpy())
            if SMOKE_TEST and batch_i >= 1:
                break
    elapsed = time.perf_counter() - start
    print(f"{'Train' if training else 'Validation'}: {count} images, {elapsed:.1f} s ({count/elapsed:.1f} images/s)")
    return (total_loss/count, correct/count,
            np.concatenate(all_probs) if all_probs else None,
            np.concatenate(all_targets) if all_targets else None)

# Establish baseline with this exact loader before fine-tuning.
baseline_loss, baseline_acc, _, _ = run_epoch(val_loader)
(OUT/'baseline_metrics.json').write_text(json.dumps({'accuracy':baseline_acc,'loss':baseline_loss}, indent=2))
history = []
best_accuracy, best_loss, stale = baseline_acc, baseline_loss, 0
baseline = dict(source_checkpoint)
baseline.update(config=config, epoch=0, val_accuracy=baseline_acc, val_loss=baseline_loss,
                source_epoch=source_checkpoint['epoch'])
torch.save(baseline, OUT/'best_model.pt')
for epoch in range(1, (2 if SMOKE_TEST else EPOCHS) + 1):
    epoch_start = time.perf_counter()
    freeze = epoch <= WARMUP_EPOCHS
    for p in model.features.parameters():
        p.requires_grad_(not freeze)
    train_loss, train_acc, _, _ = run_epoch(train_loader, True, freeze)
    val_loss, val_acc, _, _ = run_epoch(val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_acc,
                    'val_loss': val_loss, 'val_accuracy': val_acc,
                    'backbone_lr': optimizer.param_groups[0]['lr'],
                    'head_lr': optimizer.param_groups[1]['lr'],
                    'seconds': time.perf_counter() - epoch_start})
    pd.DataFrame(history).to_csv(OUT/'history.csv', index=False)
    print(f'Epoch {epoch:02d}: train loss {train_loss:.4f}, acc {train_acc:.4f} | val loss {val_loss:.4f}, acc {val_acc:.4f}')
    improved = val_acc > best_accuracy or (val_acc == best_accuracy and val_loss < best_loss)
    if improved:
        best_accuracy, best_loss, stale = val_acc, val_loss, 0
        # CPU checkpoint tensors avoid DirectML storage-serialization issues.
        torch.save({'state_dict': {k: v.detach().cpu() for k,v in model.state_dict().items()},
                    'classes': classes, 'config': config, 'epoch': epoch,
                    'val_accuracy': val_acc, 'val_loss': val_loss}, OUT/'best_model.pt')
    else:
        stale += 1
    scheduler.step(val_loss)
    # Save completed-epoch state for recovery; best_model remains separate.
    def cpu_tree(value):
        if torch.is_tensor(value): return value.detach().cpu().clone()
        if isinstance(value, dict): return {k:cpu_tree(v) for k,v in value.items()}
        if isinstance(value, list): return [cpu_tree(v) for v in value]
        if isinstance(value, tuple): return tuple(cpu_tree(v) for v in value)
        return value
    torch.save({'state_dict':cpu_tree(model.state_dict()),
                'optimizer':cpu_tree(optimizer.state_dict()),
                'scheduler':scheduler.state_dict(), 'epoch':epoch,
                'classes':classes, 'config':config, 'history':history}, OUT/'last_model.pt')
    if stale >= PATIENCE:
        print('Early stopping.'); break

In [ ]:
checkpoint=torch.load(OUT/'best_model.pt',map_location='cpu',weights_only=True)
model.load_state_dict(checkpoint['state_dict'])
val_loss,val_acc,probs,truth=run_epoch(val_loader)
pred=probs.argmax(1)
report=classification_report(truth,pred,labels=list(range(8)),target_names=classes,output_dict=True,zero_division=0)
pd.DataFrame(report).transpose().to_csv(OUT/'classification_metrics.csv')
(OUT/'validation_metrics.json').write_text(json.dumps({'loss':val_loss,'accuracy':val_acc,'checkpoint_epoch':checkpoint['epoch'],'smoke_test':SMOKE_TEST},indent=2))
results=val_df.iloc[:len(pred)][['image_id','path','class']].copy()
results['prediction']=[classes[i] for i in pred]; results['confidence']=probs.max(1)
results.to_csv(OUT/'validation_predictions.csv',index=False)
np.save(OUT/'validation_probabilities.npy',probs)
cm=confusion_matrix(truth,pred,labels=list(range(8)))
pd.DataFrame(cm,index=classes,columns=classes).to_csv(OUT/'confusion_matrix.csv')
fig,ax=plt.subplots(figsize=(9,8))
ConfusionMatrixDisplay(cm,display_labels=classes).plot(ax=ax,xticks_rotation=45,colorbar=False)
plt.tight_layout(); plt.savefig(OUT/'confusion_matrix.png',dpi=150); plt.show()
h=pd.DataFrame(history)
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,metric in zip(axes,['loss','accuracy']):
    ax.plot(h.epoch,h['train_'+metric],label='Train (augmented)')
    ax.plot(h.epoch,h['val_'+metric],label='Validation')
    ax.set(xlabel='Epoch',ylabel=metric); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'learning_curves.png',dpi=150); plt.show()
errors=results[results['class']!=results.prediction].sort_values('confidence',ascending=False)
errors.to_csv(OUT/'validation_errors.csv',index=False)
if len(errors):
    fig,axes=plt.subplots(3,4,figsize=(12,9))
    for ax in axes.flat: ax.axis('off')
    for ax,row in zip(axes.flat,errors.head(12).itertuples()):
        with Image.open(row.path) as raw: ax.imshow(ImageOps.exif_transpose(raw).convert('RGB'))
        ax.set_title(f'True: {row[3]}\nPred: {row.prediction} ({row.confidence:.2f})',fontsize=9)
    plt.tight_layout(); plt.savefig(OUT/'validation_error_examples.png',dpi=150); plt.show()
print('Validation accuracy:',val_acc,'| loss:',val_loss)

In [ ]:
if SMOKE_TEST:
    print('Smoke test finished. No submission generated. Restart with SMOKE_TEST=False and a NEW RUN_NAME for full training.')
else:
    selected = torch.load(OUT/'best_model.pt', map_location='cpu', weights_only=True)
    model.load_state_dict(selected['state_dict'])
    test_df=sample[['image_id']].copy()
    test_df['path']=resolve_images(DATA_ROOT/'test',test_df.image_id)
    test_loader=loader(test_df,eval_tf)
    model.eval(); chunks=[]
    with torch.no_grad():
        for images,_ in tqdm(test_loader,desc='Predict test'):
            logits=model(images.to(DEVICE))
            chunks.append(logits.cpu().softmax(1).numpy())
    test_probs=np.concatenate(chunks)
    assert np.isfinite(test_probs).all()
    submission=sample[['image_id','class']].copy()
    submission['class']=[classes[i] for i in test_probs.argmax(1)]
    assert len(submission)==len(sample)==29443
    assert submission.image_id.tolist()==sample.image_id.tolist()
    assert submission.image_id.is_unique and submission['class'].isin(classes).all()
    submission.to_csv(OUT/'submission.csv',index=False)
    check=pd.read_csv(OUT/'submission.csv',dtype=str,keep_default_na=False)
    assert check.equals(submission)
    np.save(OUT/'test_probabilities.npy',test_probs)
    print('Submission saved:',OUT/'submission.csv')
    print(submission.head())